In [1]:
import os
import matplotlib as mpl
import numpy as np
import pandas as pd
import scanpy as sc
import celltypist
from celltypist import models
import gc

sc.settings.verbosity = 3  # verbosity: errors (0), warnings (1), info (2), hints (3)
sc.settings.set_figure_params(dpi=80, facecolor='white', color_map='viridis')
sc.logging.print_header()

scanpy==1.9.1 anndata==0.8.0 umap==0.5.3 numpy==1.20.1 scipy==1.6.1 pandas==1.4.3 scikit-learn==0.24.1 statsmodels==0.13.2 python-igraph==0.8.3 louvain==0.7.0 leidenalg==0.8.3 pynndescent==0.5.2


In [3]:
def filter_norma(adata, min_counts=1000, min_genes=500, min_donor=50):
    #sc.pp.filter_cells(adata, min_counts=min_counts)

    sc.pp.filter_cells(adata, min_genes=min_genes)

    adata = adata[adata.obs['QC'] == 'Pass']

    # Remove donors with few than 50 cells
    donor_counts = adata.obs['donor_id'].value_counts()

    donor_counts.sort_values()[:30]

    donor_counts[donor_counts<min_donor].sum()

    adata = adata[adata.obs['donor_id'].isin( donor_counts[donor_counts>=min_donor].index)]
    
    adata.layers["counts"] = adata.X.copy()
# size and log1p normalization
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)

    adata.raw = adata  # keep full dimension safe
    return adata

#### Immune_compartment PBMC

In [4]:
path="/nfs/team205/bh14/Datasets/Remapped/raw_adata/QC_concatenated/"

In [5]:
adata = sc.read_h5ad(path+'Immune_compartment_Concatenated_dataset.h5ad')

In [6]:
adata.obs['dataset_id'].value_counts()

GSE157278_Hong_2020            60205
GSE163314_Lefferts_2021        53037
E-MTAB-9492_Penkava_2020       33698
GSE134809_Martin_et_al_2019    26855
Braga_asthma_CD4                 929
Name: dataset_id, dtype: int64

In [7]:
adata.shape

(174724, 36601)

In [8]:
adata = filter_norma(adata=adata, min_genes=500, min_donor=50)

filtered out 12722 cells that have less than 500 genes expressed
normalizing counts per cell
    finished (0:00:00)


In [9]:
adata.shape

(156806, 36601)

In [10]:
adata.obs['dataset_id'].value_counts()

GSE157278_Hong_2020            54255
GSE163314_Lefferts_2021        47981
E-MTAB-9492_Penkava_2020       33020
GSE134809_Martin_et_al_2019    20751
Braga_asthma_CD4                 799
Name: dataset_id, dtype: int64

In [11]:
adata.X.expm1().sum(axis = 1)

matrix([[10000.   ],
        [10000.001],
        [10000.   ],
        ...,
        [10000.   ],
        [10000.   ],
        [10000.   ]], dtype=float32)

In [12]:
adata.write_h5ad(path+'Immune_compartment_QC_filtered_2.h5ad')

#### Immune_compartment filtred from tissue

In [16]:
path="/nfs/team205/bh14/Datasets/Remapped/raw_adata/QC_concatenated/"

In [17]:
adata = sc.read_h5ad(path+'Immune_compartment_sorted_from_Tissue_Concatenated_dataset.h5ad')

In [18]:
adata.obs['dataset_id'].value_counts()

E-MTAB-8322_Stefeno_2020    80142
GSE121380_Huang_2019        50287
E-MTAB-9492_Penkava_2020    41089
GSE161500_Abji_2020          2718
Braga_asthma_CD4             1708
Name: dataset_id, dtype: int64

In [19]:
adata.shape

(175944, 36601)

In [20]:
adata = filter_norma(adata=adata, min_genes=500, min_donor=50)

filtered out 20337 cells that have less than 500 genes expressed
normalizing counts per cell
    finished (0:00:00)


In [21]:
adata.shape

(148806, 36601)

In [22]:
adata.obs['dataset_id'].value_counts()

E-MTAB-8322_Stefeno_2020    66531
E-MTAB-9492_Penkava_2020    39512
GSE121380_Huang_2019        38849
GSE161500_Abji_2020          2263
Braga_asthma_CD4             1651
Name: dataset_id, dtype: int64

In [23]:
adata.X.expm1().sum(axis = 1)

matrix([[ 9999.999],
        [ 9999.998],
        [10000.   ],
        ...,
        [10000.001],
        [10000.   ],
        [ 9999.999]], dtype=float32)

In [24]:
adata.write_h5ad(path+'Immune_compartment_sorted_from_Tissue_QC_filtered_2.h5ad')

In [25]:
import gc
del adata
gc.collect()

12636

#### Tissue

In [26]:
path="/nfs/team205/bh14/Datasets/Remapped/raw_adata/QC_concatenated/"

In [27]:
adata = sc.read_h5ad(path+'Tissue_or_Nonimmune_Concatenated_dataset.h5ad')

In [28]:
adata.obs['dataset_id'].value_counts()

E-MTAB-8142_Reynolds_2021_Dermis       185982
SDY1765_Cross_tissue_stromal           156107
GSE134809_Martin_et_al_2019            102215
E-MTAB-8142_Reynolds_2021_Epidermis    100445
Elmentaite_2020                         73614
SDY1599_Wei_2020                        26472
GSE163314_Lefferts_2021                 14559
GSE176308_Nanus_2021                     4248
GSE121380_Huang_2019                     2178
Name: dataset_id, dtype: int64

In [29]:
adata.shape

(665820, 36601)

In [30]:
adata = filter_norma(adata=adata, min_genes=500, min_donor=50)

filtered out 82718 cells that have less than 500 genes expressed
normalizing counts per cell
    finished (0:00:02)


In [31]:
adata.shape

(547633, 36601)

In [32]:
gc.collect()

1686

In [33]:
adata.obs['dataset_id'].value_counts()

E-MTAB-8142_Reynolds_2021_Dermis       184259
SDY1765_Cross_tissue_stromal           121162
E-MTAB-8142_Reynolds_2021_Epidermis     99114
GSE134809_Martin_et_al_2019             56829
Elmentaite_2020                         48053
SDY1599_Wei_2020                        23128
GSE163314_Lefferts_2021                 10453
GSE176308_Nanus_2021                     4171
GSE121380_Huang_2019                      464
Name: dataset_id, dtype: int64

In [34]:
adata.X.expm1().sum(axis = 1)

matrix([[10000.   ],
        [10000.   ],
        [10000.   ],
        ...,
        [10000.   ],
        [10000.001],
        [10000.001]], dtype=float32)

In [35]:
adata.write_h5ad(path+'Tissue_or_Nonimmune_QC_filtered_2.h5ad')

In [36]:
del adata
gc.collect()

2017

#### Tissue sorted and non-sorted

In [37]:
path="/nfs/team205/bh14/Datasets/Remapped/raw_adata/QC_concatenated/"

In [38]:
adata = sc.read_h5ad(path+'All_from_Tissue_Concatenated.h5ad')

In [39]:
adata.obs['dataset_id'].value_counts()

E-MTAB-8142_Reynolds_2021_Dermis       185982
SDY1765_Cross_tissue_stromal           156107
GSE134809_Martin_et_al_2019            102215
E-MTAB-8142_Reynolds_2021_Epidermis    100445
E-MTAB-8322_Stefeno_2020                80142
Elmentaite_2020                         73614
GSE121380_Huang_2019                    52465
E-MTAB-9492_Penkava_2020                41089
SDY1599_Wei_2020                        26472
GSE163314_Lefferts_2021                 14559
GSE176308_Nanus_2021                     4248
GSE161500_Abji_2020                      2718
Braga_asthma_CD4                         1708
Name: dataset_id, dtype: int64

In [40]:
adata.shape

(841764, 36601)

In [41]:
adata = filter_norma(adata=adata, min_genes=500, min_donor=50)

filtered out 103055 cells that have less than 500 genes expressed
normalizing counts per cell
    finished (0:00:03)


In [42]:
adata.shape

(696439, 36601)

In [43]:
gc.collect()

2129

In [44]:
adata.obs['dataset_id'].value_counts()

E-MTAB-8142_Reynolds_2021_Dermis       184259
SDY1765_Cross_tissue_stromal           121162
E-MTAB-8142_Reynolds_2021_Epidermis     99114
E-MTAB-8322_Stefeno_2020                66531
GSE134809_Martin_et_al_2019             56829
Elmentaite_2020                         48053
E-MTAB-9492_Penkava_2020                39512
GSE121380_Huang_2019                    39313
SDY1599_Wei_2020                        23128
GSE163314_Lefferts_2021                 10453
GSE176308_Nanus_2021                     4171
GSE161500_Abji_2020                      2263
Braga_asthma_CD4                         1651
Name: dataset_id, dtype: int64

In [45]:
adata.X.expm1().sum(axis = 1)

matrix([[ 9999.999],
        [ 9999.998],
        [10000.   ],
        ...,
        [10000.   ],
        [10000.001],
        [10000.001]], dtype=float32)

In [46]:
adata.write_h5ad(path+'All_from_Tissue_QC_filtered_2.h5ad')